# Prédiction d'affluence — validations Navigo journalières

**Cible** : `VALD_NAVIGO` (nombre de validations Forfait Navigo par jour sur le réseau ferré francilien).

**Pourquoi ce modèle complète le précédent** :

| Modèle | Cible | Couverture du cahier des charges |
|---|---|---|
| Hybride ARIMAX+LSTM (déjà fait) | NO2 sur le périph | Prédiction pollution + proxy trafic auto |
| **Ce notebook** | **VALD_NAVIGO journalier** | **Prédiction affluence transport en commun** |

Ensemble, les deux modèles permettent d'anticiper le **report modal** (chute validations FER → hausse NO2 sur le périph).

**Granularité** : journalière (le NO2 est horaire, mais les validations sont fournies par jour).
La granularité fine n'a pas de sens pour ce signal.

**Architecture** : même approche que le notebook précédent, adaptée au journalier :
1. **SARIMAX** (= ARIMAX + saisonnalité hebdomadaire) pour capturer le pattern lundi-dimanche
2. **LSTM** sur les résidus pour les patterns non-linéaires

**Étapes** :
1. Préparation (dédoublonnage journalier)
2. Split temporel
3. Vérification stationnarité + saisonnalité hebdomadaire
4. Pipeline SARIMAX + LSTM
5. Évaluation comparée (baseline / SARIMAX / hybride)
6. Visualisations
7. Sauvegarde

In [ ]:
import os, pickle, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

DATA_DIR   = Path("../data/processed")
MODELS_DIR = Path("../src/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Étape 1 — Préparation du dataset journalier

Le `dataset_entrainement_2024.csv` a une granularité horaire × segment (70 272 lignes).
Mais `VALD_NAVIGO` est **journalier** : c'est la même valeur répétée 192 fois par jour
(24 heures × 8 segments) par broadcast.

On déduplique pour revenir à **366 lignes** (une par jour), sinon le modèle se fait
flouer par les répétitions artificielles.

In [ ]:
df_raw = pd.read_csv(DATA_DIR / "dataset_entrainement_2024.csv")

# Dédoublonnage : une seule ligne par jour
daily = df_raw.drop_duplicates(subset="DATE").copy()
daily["DATE"] = pd.to_datetime(daily["DATE"])
daily = daily.sort_values("DATE").reset_index(drop=True)

# On retire les colonnes horaires/segment qui n'ont plus de sens au niveau journalier
cols_a_jeter = ["time", "segment", "HEURE", "HEURE_SIN", "HEURE_COS", "PERIODE",
                "NO2", "PM10", "PM25"]
daily = daily.drop(columns=cols_a_jeter)

print(f"Dataset horaire : {df_raw.shape}")
print(f"Dataset journalier dédoublonné : {daily.shape}  (attendu : 366 × ~38)")
print(f"\nPériode : {daily['DATE'].min().date()} → {daily['DATE'].max().date()}")
daily.head(3)

## Étape 2 — Split temporel

Mêmes coupures que pour le modèle NO2 (cohérence entre les deux modèles).

In [ ]:
train = daily[daily["DATE"] <  "2024-09-01"].copy()
val   = daily[(daily["DATE"] >= "2024-09-01") & (daily["DATE"] < "2024-11-01")].copy()
test  = daily[daily["DATE"] >= "2024-11-01"].copy()

print(f"Train : {len(train):>4} jours  ({train['DATE'].min().date()} → {train['DATE'].max().date()})")
print(f"Val   : {len(val):>4} jours  ({val['DATE'].min().date()} → {val['DATE'].max().date()})")
print(f"Test  : {len(test):>4} jours  ({test['DATE'].min().date()} → {test['DATE'].max().date()})")

## Étape 3 — Diagnostic de la série

Trois vérifications pour justifier le choix SARIMAX :

1. **Stationnarité** (test ADF) — détermine `d` dans SARIMA(p,d,q)
2. **Saisonnalité hebdomadaire** (moyenne par jour de la semaine) — justifie l'usage de
   SARIMAX au lieu d'ARIMAX simple
3. **Visualisation** de la série pour repérer les ruptures (vacances, JO, etc.)

In [ ]:
pval = adfuller(train["VALD_NAVIGO"])[1]
print(f"Test ADF : p-value = {pval:.4f}")
print(f"  → {'série stationnaire (d=0)' if pval < 0.05 else 'différenciation requise (d=1)'}")

# Saisonnalité hebdomadaire
mean_dow = train.groupby("JOUR_SEMAINE")["VALD_NAVIGO"].mean()
jours = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
print(f"\nMoyenne validations par jour de la semaine (en millions) :")
for i, m in mean_dow.items():
    print(f"  {jours[i]} : {m/1e6:.2f} M")

In [ ]:
# Visualisation de la série complète
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train["DATE"], train["VALD_NAVIGO"]/1e6, label="Train", linewidth=0.9)
ax.plot(val["DATE"],   val["VALD_NAVIGO"]/1e6,   label="Validation", linewidth=0.9)
ax.plot(test["DATE"],  test["VALD_NAVIGO"]/1e6,  label="Test", linewidth=0.9)
ax.set_xlabel("Date")
ax.set_ylabel("Validations Navigo (millions)")
ax.set_title("VALD_NAVIGO — série complète 2024")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture** :

- Le cycle hebdomadaire est très visible (oscillations rapides).
- Les creux importants correspondent aux vacances scolaires et fériés.
- Au-delà de juillet, les JO 2024 ont fortement perturbé l'affluence — pattern atypique.

Le test ADF est positif (stationnaire à 5 %) et la saisonnalité hebdomadaire est nette.
→ On utilisera **SARIMAX(1,0,1)(1,1,1,7)** : ordres non-saisonniers simples + composante
saisonnière de période 7 jours.

## Étape 4 — Pipeline SARIMAX + LSTM

Mêmes principes que le notebook NO2, adaptés au journalier :

- **Features exogènes** : on garde la météo journalière et les flags calendrier (JO,
  fériés, vacances, jour non-ouvré). On retire `JOUR_SEMAINE` puisque la composante
  saisonnière du SARIMAX la capture déjà.
- **LSTM look-back = 7 jours** (une semaine d'historique de résidus). C'est cohérent
  avec la saisonnalité hebdomadaire.
- **Réseau plus petit** que pour le NO2 (32 unités au lieu de 50) : on n'a que 244 jours
  de train, un modèle trop gros surapprendrait.

In [ ]:
EXOG_FEATURES = [
    # Calendrier
    "JOUR_FERIE", "VACANCES_SCOLAIRES", "JO", "JOP",
    "JOUR_NON_OUVRE", "JOUR_PERTURBE",
    # Météo
    "TEMP_AVG_C", "WINDSPEED_MAX_KMH", "PRECIP_TOTAL_DAY_MM",
    "HUMIDITY_MAX_PERCENT", "PRESSURE_MAX_MB", "CLOUDCOVER_AVG_PERCENT",
]

LOOK_BACK = 7   # une semaine

# 1. Standardisation des exogènes (fit sur train uniquement)
scaler_exog = StandardScaler()
exog_tr = pd.DataFrame(scaler_exog.fit_transform(train[EXOG_FEATURES]),
                       columns=EXOG_FEATURES, index=train.index)
exog_va = pd.DataFrame(scaler_exog.transform(val[EXOG_FEATURES]),
                       columns=EXOG_FEATURES, index=val.index)
exog_te = pd.DataFrame(scaler_exog.transform(test[EXOG_FEATURES]),
                       columns=EXOG_FEATURES, index=test.index)

# 2. SARIMAX
print("Entraînement SARIMAX(1,0,1)(1,1,1,7) ...")
sarimax = SARIMAX(
    train["VALD_NAVIGO"].values,
    exog=exog_tr.values,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
).fit(disp=False)

print(f"  AIC = {sarimax.aic:.0f}")

# Forecasts
pred_tr_sarimax = sarimax.fittedvalues
pred_va_sarimax = sarimax.forecast(steps=len(val),  exog=exog_va.values)
pred_te_sarimax = sarimax.forecast(steps=len(test), exog=exog_te.values)

In [ ]:
# 3. Résidus
res_tr = train["VALD_NAVIGO"].values - pred_tr_sarimax
res_va = val["VALD_NAVIGO"].values   - pred_va_sarimax
res_te = test["VALD_NAVIGO"].values  - pred_te_sarimax

# 4. LSTM sur résidus (scaler fit sur train uniquement)
scaler_res = MinMaxScaler()
res_tr_s = scaler_res.fit_transform(res_tr.reshape(-1,1)).flatten()
res_va_s = scaler_res.transform(res_va.reshape(-1,1)).flatten()
res_te_s = scaler_res.transform(res_te.reshape(-1,1)).flatten()

def make_sequences(arr, lb):
    X, y = [], []
    for i in range(len(arr) - lb):
        X.append(arr[i:i+lb])
        y.append(arr[i+lb])
    return np.array(X).reshape(-1, lb, 1), np.array(y)

Xtr, ytr = make_sequences(res_tr_s, LOOK_BACK)
Xva, yva = make_sequences(res_va_s, LOOK_BACK)
Xte, yte = make_sequences(res_te_s, LOOK_BACK)

print(f"Séquences : train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}")

lstm = Sequential([
    LSTM(32, return_sequences=True, input_shape=(LOOK_BACK, 1)),
    Dropout(0.2),
    LSTM(32),
    Dense(1),
])
lstm.compile(optimizer="adam", loss="mse")
es = EarlyStopping(patience=5, restore_best_weights=True)

print("\nEntraînement LSTM sur résidus...")
hist = lstm.fit(Xtr, ytr, validation_data=(Xva, yva),
                epochs=50, batch_size=8, callbacks=[es], verbose=0)
print(f"  {len(hist.history['loss'])} epochs (early stopping)")

In [ ]:
# 5. Prédiction hybride sur test
pred_res_te_s = lstm.predict(Xte, verbose=0).flatten()
pred_res_te   = scaler_res.inverse_transform(pred_res_te_s.reshape(-1,1)).flatten()

# Alignement : on perd les LOOK_BACK premiers jours du test
y_test_align    = test["VALD_NAVIGO"].values[LOOK_BACK:]
sarimax_align   = pred_te_sarimax[LOOK_BACK:]
hybrid_align    = sarimax_align + pred_res_te
dates_align     = test["DATE"].values[LOOK_BACK:]

print(f"Test set utilisable : {len(y_test_align)} jours")

## Étape 5 — Évaluation comparée

Trois modèles :
- **Baseline** : moyenne par jour de la semaine (calculée sur le train).
- **SARIMAX seul**
- **SARIMAX + LSTM (hybride)**

Les chiffres sont exprimés en **millions de validations**.

In [ ]:
# Baseline : moyenne par jour de la semaine
bow = train.groupby("JOUR_SEMAINE")["VALD_NAVIGO"].mean()
baseline_test_full = test["JOUR_SEMAINE"].map(bow).values
baseline_align     = baseline_test_full[LOOK_BACK:]

def metrics(y, yp):
    return {
        "RMSE (M)": np.sqrt(mean_squared_error(y, yp)) / 1e6,
        "MAE (M)":  mean_absolute_error(y, yp) / 1e6,
        "R²":       r2_score(y, yp),
        "MAPE (%)": np.mean(np.abs((y - yp) / y)) * 100,
    }

results = pd.DataFrame({
    "Baseline (DoW)":   metrics(y_test_align, baseline_align),
    "SARIMAX":          metrics(y_test_align, sarimax_align),
    "SARIMAX + LSTM":   metrics(y_test_align, hybrid_align),
}).T.round(3)

print(results.to_string())

# Gain final
gain_rmse = (results.loc["Baseline (DoW)", "RMSE (M)"] - results.loc["SARIMAX + LSTM", "RMSE (M)"]) / results.loc["Baseline (DoW)", "RMSE (M)"] * 100
print(f"\nGain RMSE hybride vs baseline : -{gain_rmse:.1f} %")

## Étape 6 — Visualisations

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
dates = pd.to_datetime(dates_align)
ax.plot(dates, y_test_align/1e6,  label="Observé",        linewidth=2, color="black")
ax.plot(dates, sarimax_align/1e6, label="SARIMAX seul",   linewidth=1.5, alpha=0.7)
ax.plot(dates, hybrid_align/1e6,  label="SARIMAX + LSTM", linewidth=1.5, alpha=0.9, color="red")
ax.plot(dates, baseline_align/1e6, label="Baseline (DoW)", linewidth=1, alpha=0.5, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Validations Navigo (millions)")
ax.set_title("Prédictions sur le test set (nov-déc 2024)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Erreurs résiduelles du modèle hybride
errors = y_test_align - hybrid_align

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramme des erreurs
axes[0].hist(errors/1e6, bins=20, edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--", label="0")
axes[0].set_xlabel("Erreur (millions)")
axes[0].set_ylabel("Fréquence")
axes[0].set_title("Distribution des erreurs résiduelles")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Prédit vs observé
axes[1].scatter(y_test_align/1e6, hybrid_align/1e6, alpha=0.6)
lim_min = min(y_test_align.min(), hybrid_align.min()) / 1e6
lim_max = max(y_test_align.max(), hybrid_align.max()) / 1e6
axes[1].plot([lim_min, lim_max], [lim_min, lim_max], "r--", label="Prédiction parfaite")
axes[1].set_xlabel("Observé (millions)")
axes[1].set_ylabel("Prédit (millions)")
axes[1].set_title("Prédit vs observé — modèle hybride")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Étape 7 — Sauvegarde du modèle

In [ ]:
prefix = "navigo_daily"

with open(MODELS_DIR / f"{prefix}_sarimax.pkl", "wb") as f:
    pickle.dump(sarimax, f)
lstm.save(MODELS_DIR / f"{prefix}_lstm.keras")
joblib.dump(scaler_exog, MODELS_DIR / f"{prefix}_scaler_exog.joblib")
joblib.dump(scaler_res,  MODELS_DIR / f"{prefix}_scaler_res.joblib")

print(f"✓ Modèles sauvegardés dans {MODELS_DIR}/")
for f in sorted(MODELS_DIR.glob(f"{prefix}*")):
    print(f"  - {f.name}")

---

## Récap pour le mémoire

Ce notebook **complète** le modèle NO2 en répondant explicitement au besoin de
prédiction d'affluence transport mentionné dans le cahier des charges
(section IV : *"Optimisation des transports publics : modèles prédictifs"*).

**Architecture identique** au modèle NO2 : SARIMAX + LSTM en hybride, garantissant la
cohérence méthodologique entre les deux prédictions.

**Conformité au cahier des charges — récapitulatif** :

| Besoin | Réponse |
|---|---|
| Anticiper les flux de trafic | Modèle NO2 (proxy du trafic auto) |
| Optimiser les transports publics | **Modèle VALD_NAVIGO (ce notebook)** |
| Modèles hybrides ARIMA + DL | ✓ SARIMAX + LSTM |
| Modèles prédictifs (pluriel) | XGBoost + Hybride NO2 + Hybride Navigo |
| Métriques de performance | RMSE / MAE / R² / MAPE |

**Lecture du résultat** : la combinaison des deux prédictions permet d'anticiper le
**report modal** (chute d'affluence FER + hausse de NO2 sur le périph = signature de
perturbation transport en commun).